# Importation des modules

In [1]:
import pandas as pd
import numpy as np
import statsmodels

# Chargement des données
diviser les dépenses de santé par le PIB pour les avoir en %  
rajouter les données pour 1994 - 2004 en regardant sur les données de l'ocde
rajouter la base mortalite  
faire le tri dans les pays de la bdd nb de médecins  


In [3]:
def load_data(depenses_sante, PIB, pop_65, pop_tot, densite_medicale, out_of_pocket, mortalite) :  #à remplacer par load_data() quand ce sera codé
    # depenses_sante = pd.read_csv("...")
    # ...
    return depenses_sante, PIB, pop_65, pop_tot, densite_medicale, out_of_pocket, mortalite

#load_data()

# Construction du Time-to-Death

In [5]:
def time_to_death(mortalite, pop) :
    # pour pop, à voir si on prend pop_tot, pop_65 ou une population plus âgée encore

    # fait par chatgpt, à revoir en fonction de la structure des données
    df = mortalite.merge(
        pop_tot,
        on=["country", "year", "age"],
        how="inner"
    )

    df["weighted_mortality"] = df["mortality_rate"] * df["population"]
    ttd = (
        df.groupby(["country", "year"])["weighted_mortality"]
          .sum()
          .reset_index(name="TTD")
    )
    return ttd

#pop = pop_65   # ou pop_80, ou pop_tot...
#ttd = time_to_death(mortalite, pop)

# Mise en forme du panel

In [7]:
def panel(depenses_sante, pib, pop_65, pop_tot, densite_medicale, oop, ttd) :
    #construction du panel contenant toutes les données dont on a besoin dans un seul panel
    panel = (
        depenses_sante.merge(pop_65, on=["country", "year"])
           .merge(pib, on=["country", "year"])
           .merge(densite_medicale, on=["country", "year"])
           .merge(oop, on=["country", "year"])
           .merge(ttd, on=["country", "year"])
    )

    panel = panel.sort_values(["country", "year"])
    panel.set_index(["country", "year"], inplace=True)
    return panel

#panel = panel(depenses_sante, pib, pop_65, pop_tot, densite_medicale, oop, ttd)

# Régression et GMM